### RAG pipeline

#### Data ingestion + chunking

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import pandas as pd

import os
from pathlib import Path

# The below code is used to access packages from the root
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from app.agent.rag.ingestion.data_ingestion import process_and_load_file
from app.agent.rag.ingestion.embeddings import get_embedding_model, generate_embeddings

In [3]:
print(os.getcwd())

c:\Users\USER\Documents\AI projects\Denseless-local\app\agent


In [ ]:
# Upload your pdf material to "app\pdfs" dir
# Insert its path as the parameter
docs = process_and_load_file("../pdfs/Disaster Recovery Plan.pdf")
print(docs)

PDF page count: 4
Processing: Disaster Recovery Plan.pdf


INFO: split_pdf event=plan_created operation_id=f982a8cc-9e4a-4169-a3c2-2e69b5ddee5e filename=Disaster Recovery Plan.pdf strategy=hi_res page_range=1-4 page_count=4 split_size=2 chunk_count=2 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset
INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=f982a8cc-9e4a-4169-a3c2-2e69b5ddee5e chunk_count=2 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_complete operation_id=f982a8cc-9e4a-4169-a3c2-2e69b5ddee5e chunk_count=2 success_count=2 failure_count=0 transport_failure_count=0 elapsed_ms=10358 allow_failed=False


  → Parsed into 49 document(s)
[Document(metadata={'section': 'DISASTER RECOVERY PLAN', 'category': 'Title', 'page_number': 1, 'element_id': 'd106a9434f1493646b522a2a746f4eb0', 'filename': 'Disaster Recovery Plan.pdf'}, page_content='DISASTER RECOVERY PLAN'), Document(metadata={'section': 'What is a Disaster Recovery Plan', 'category': 'Title', 'page_number': 1, 'element_id': 'c26c0fdcccd24b498276424d943d3841', 'filename': 'Disaster Recovery Plan.pdf'}, page_content='What is a Disaster Recovery Plan?'), Document(metadata={'section': 'What is a Disaster Recovery Plan', 'category': 'NarrativeText', 'page_number': 1, 'element_id': 'd1e6eacc5a5fe037fdadc5b3c0e8f127', 'filename': 'Disaster Recovery Plan.pdf'}, page_content='Disaster recovery (DR) is an organization’s ability to restore access and functionality to IT infrastructure after a disaster event, whether natural or caused by human action (or error). DR is considered a subset of business continuity, explicitly focusing on ensuring th

In [6]:
pd.set_option('display.max_rows', None)

df = pd.DataFrame([
    {**doc.metadata, "content_preview": doc.page_content[:300], "character count": len(doc.page_content)}
    for doc in docs
])
# df

In [7]:
df.head

<bound method NDFrame.head of                                               section       category  \
0   CSC 315 : COMPUTER ARCHITECTURE AND ORGANIZATI...          Title   
1                                Week One: Lesson One          Title   
2                                        Introduction       ListItem   
3                                        Introduction       ListItem   
4                                        Introduction       ListItem   
5                   1.1 Organization and Architecture          Title   
6                   1.1 Organization and Architecture  NarrativeText   
7                   1.1 Organization and Architecture  NarrativeText   
8                   1.1 Organization and Architecture  NarrativeText   
9                   1.1 Organization and Architecture  NarrativeText   
10                  1.1 Organization and Architecture  NarrativeText   
11                  1.1 Organization and Architecture  NarrativeText   
12                  1.1 Organizati

In [8]:
pd.set_option('display.max_rows', 40)
print(df)

                                              section       category  \
0   CSC 315 : COMPUTER ARCHITECTURE AND ORGANIZATI...          Title   
1                                Week One: Lesson One          Title   
2                                        Introduction       ListItem   
3                                        Introduction       ListItem   
4                                        Introduction       ListItem   
..                                                ...            ...   
83                       1.3 Von Neumann architecture  NarrativeText   
84                             Von Neumann bottleneck          Title   
85                             Von Neumann bottleneck  NarrativeText   
86                             Von Neumann bottleneck  NarrativeText   
87                             Von Neumann bottleneck  NarrativeText   

    page_number                        element_id                  filename  \
0             1  d69926b97dc4ad40e0754c8ed3bc844f  CSC 3

In [8]:
len(docs[11].page_content)

276

#### Embeddings

In [ ]:
embedder = get_embedding_model()
vectors = generate_embeddings(docs, embedder)
print(vectors)

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


Loading embedding model: BAAI/bge-small-en-v1.5 (device=cpu)


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BA

  → Embedding model loaded successfully. Embedding dimension: 384
Generating embeddings for 49 document chunk(s)...


Embedding documents: 100%|██████████| 2/2 [00:01<00:00,  1.24it/s]

  → Successfully generated 49 embedding vectors.
  → Embeddings shape: (49, 384)
[[-0.02354764 -0.03415421  0.03727565 ...  0.02472775  0.03189892
   0.00855878]
 [-0.02427611 -0.0350576   0.02377369 ...  0.02949315  0.03969157
  -0.01117094]
 [-0.04075452  0.0035219   0.05269271 ...  0.00700633  0.01639645
  -0.01019619]
 ...
 [-0.06215926  0.00825759  0.00255826 ...  0.04869787  0.00376491
  -0.00804847]
 [-0.07669389  0.01508064  0.03468475 ...  0.0276745   0.02580342
  -0.01168488]
 [ 0.00017653  0.03075893  0.04102878 ...  0.01552661  0.06049531
   0.00567352]]


#### Vector store

In [33]:
from app.agent.rag.ingestion.vector_store import get_vector_store, add_documents_to_chroma

In [ ]:
# Insert any number as the student id; if it doesn't exist,
# it will be created. 
# It is advised to clear the "app\chroma_db" dir's contents
# each time this notebook is run
store = get_vector_store(student_id="1019", embedder=embedder)

Vector store backend: 'chroma' | student: '1019'
Initialising Chroma store — student: '1019'
  → Collection : student_1019
  → Persist dir: ../chroma_db
  → Chroma store ready. 
Documents in collection: 352


In [ ]:
# Upon successful creation of vector store, you can add the 
# document's embeddings and chunks to the db using the 
# function below by filling its parameters
add_documents_to_chroma(store, vectors, docs, False, "PQ", "Disaster Recovery Plan", "Disaster Recovery Plan")

  [vector_store] 352 existing chunk(s) found in collection.
  [vector_store] Adding 49 new chunk(s) to Chroma store...
  [vector_store] ✓ Successfully added 49 chunk(s).
  [vector_store] Total documents in collection: 401


In [ ]:
# You can add multiple docs
add_documents_to_chroma(store, vectors, docs, False, "CSC 403", "Software Engineering Intro", "Software Engineering Intro")

  [vector_store] 49 existing chunk(s) found in collection.
  [vector_store] 49 duplicate chunk(s) skipped.
  [vector_store] ✓ All documents already exist in store — nothing to add.


### Chains

You can now interact with the chains to perform the ai's various tasks.

#### Notes chain

In [ ]:
from app.agent.rag.chains.notes_chain import run_notes_chain
from app.agent.rag.retrieval.retriever import get_topic_chunks
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# Instantiate your llm (local or paid)
# local:
# llm = ChatOllama(model="gemma3:1b")
load_dotenv()
api_key = os.environ.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",  
    project="denseless",       
    location="us-central1",             
    vertexai=True,
)

INFO:google_genai._api_client:The user provided project/location will take precedence over the Vertex AI API key from the environment variable.


In [ ]:
# Fetch all the topic's chunks for notes generation by supplying
# the parameters in the get_topic_chunks function
chunks_topic =  get_topic_chunks(store, "Software Engineering Intro", "CSC 403")

In [52]:
# Case: A fast student
pdf_path = run_notes_chain(
      student_id    = "student_1019",
      topic_map     = chunks_topic,
      current_topic = "Software Engineering Intro",
      weak_topics=[],
      strong_topics = [],
      course="CSC 403",
      learning_pace = "fast",
      llm           = llm,
      embedder      = embedder,
      store         = store,
  )

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


  [token_service] Tokens remaining for 'student_1019': 9,697,183
  [token_guard] Checking tokens — student: student_1019 | remaining: 9697183 | chain: run_notes_chain

[notes_chain] ═══ Starting notes generation ═══
[notes_chain] Mode: Gemini API
[notes_chain] Student: student_1019 | Topic: Software Engineering Intro | Pace: fast
[notes_chain] Sections to process: 10

[notes_chain] Processing section 1/10: 'Course Contents'
[qa_chain] LTM store loaded for student 'student_1019' (3 memories).


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'Course Contents' condensed.
[notes_chain] ⏳ Throttle: sleeping 5s before next section.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.


...

[notes_chain] ✓ Section 'The key differences are' condensed.
[notes_chain] ⏳ Throttle: sleeping 5s before next section.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.



[notes_chain] Processing section 10/10: 'Importance of Software'
[qa_chain] LTM store loaded for student 'student_1019' (3 memories).


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'Importance of Software' condensed.
[notes_chain] PDF built: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Software_Engineering_Intro_20260507_064406.pdf

[notes_chain] ═══ Notes generation complete ═══
[notes_chain] PDF: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Software_Engineering_Intro_20260507_064406.pdf
[notes_chain] Sections completed: 10/10
[notes_chain] Total tokens — in: 10,246 | out: 1,780 | total: 12,026
  [token_service] Deducted 12,026 tokens — student: 'student_1019' | chain: run_notes_chain | in: 10,246 | out: 1,780 | remaining: 9,685,157
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 12026 | remaining: 9685157


In [53]:
# Case: An avg student
pdf_path = run_notes_chain(
      student_id    = "student_1019",
      topic_map     = chunks_topic,
      current_topic = "Software Engineering Intro",
      weak_topics=[],
      strong_topics = [],
      course="CSC 403",
      learning_pace = "average",
      llm           = llm,
      embedder      = embedder,
      store         = store,
  )

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


  [token_service] Tokens remaining for 'student_1019': 9,685,157
  [token_guard] Checking tokens — student: student_1019 | remaining: 9685157 | chain: run_notes_chain

[notes_chain] ═══ Starting notes generation ═══
[notes_chain] Mode: Gemini API
[notes_chain] Student: student_1019 | Topic: Software Engineering Intro | Pace: average
[notes_chain] Sections to process: 10

[notes_chain] Processing section 1/10: 'Course Contents'
[qa_chain] LTM store loaded for student 'student_1019' (3 memories).


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'Course Contents' condensed.
[notes_chain] ⏳ Throttle: sleeping 5s before next section.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.


...

[notes_chain] ✓ Section 'The key differences are' condensed.
[notes_chain] ⏳ Throttle: sleeping 5s before next section.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.



[notes_chain] Processing section 10/10: 'Importance of Software'
[qa_chain] LTM store loaded for student 'student_1019' (3 memories).


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'Importance of Software' condensed.
[notes_chain] PDF built: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Software_Engineering_Intro_20260507_065139.pdf

[notes_chain] ═══ Notes generation complete ═══
[notes_chain] PDF: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Software_Engineering_Intro_20260507_065139.pdf
[notes_chain] Sections completed: 10/10
[notes_chain] Total tokens — in: 10,304 | out: 2,312 | total: 12,616
  [token_service] Deducted 12,616 tokens — student: 'student_1019' | chain: run_notes_chain | in: 10,304 | out: 2,312 | remaining: 9,672,541
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 12616 | remaining: 9672541


In [26]:
# Case: A fast student
pdf_path = run_notes_chain(
      student_id    = "student_1019",
      topic_map     = chunks_topic,
      current_topic = "Interconnection Structures",
      weak_topics=[],
      strong_topics = [],
      course="CSC 315",
      learning_pace = "fast",
      llm           = llm,
      embedder      = embedder,
      store         = store,
  )

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


  [token_service] Tokens remaining for 'student_1019': 9,848,255
  [token_guard] Checking tokens — student: student_1019 | remaining: 9848255 | chain: run_notes_chain

[notes_chain] ═══ Starting notes generation ═══
[notes_chain] Mode: Gemini API
[notes_chain] Student: student_1019 | Topic: Interconnection Structures | Pace: fast
[notes_chain] Sections to process: 15

[notes_chain] Processing section 1/15: 'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK'


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK' condensed.
[notes_chain] ⏳ Throttle: sleeping 5s before next section.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.


...

[notes_chain] ✓ Section 'Word length' condensed.
[notes_chain] ⏳ Throttle: sleeping 5s before next section.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.



[notes_chain] Processing section 15/15: 'INSTRUCTION SEQUENCING'


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'INSTRUCTION SEQUENCING' condensed.
[notes_chain] PDF built: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Interconnection_Structures_20260428_224202.pdf

[notes_chain] ═══ Notes generation complete ═══
[notes_chain] PDF: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Interconnection_Structures_20260428_224202.pdf
[notes_chain] Sections completed: 15/15
[notes_chain] Total tokens — in: 15,872 | out: 3,427 | total: 19,299
  [token_service] Deducted 19,299 tokens — student: 'student_1019' | chain: run_notes_chain | in: 15,872 | out: 3,427 | remaining: 9,828,956
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 19299 | remaining: 9828956


### QA chain

In [ ]:
from app.agent.rag.chains.qa_chain import run_qa_chain, list_conversations, view_ltm
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama.chat_models import ChatOllama
from dotenv import load_dotenv
import json

In [ ]:
# Instantiate your llm (local or paid)

# llm = ChatOllama(model="mistral:latest")
load_dotenv()
api_key = os.environ.get('GOOGLE_API_KEY')

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", 
    project="denseless",       
    location="us-central1",              
    vertexai=True,
)

INFO:google_genai._api_client:The user provided project/location will take precedence over the Vertex AI API key from the environment variable.


In [ ]:
# Check out the list of available convos created in the past
# Pass the parameters for qa_chain

print(list_conversations("student_1019"))
response = run_qa_chain(
    "student_1019",
    "I dislikes analogies and finds them condescending. I prefer straight to the point, technical explanations without illustrative comparisons. Also, prefers concise responses. I get disengaged when answers are padded with background context I did not ask for. Lastly, I consistently confuse RTO (Recovery Time Objective) and RPO (Recovery Point Objective). I have mixed them up across multiple past interactions",
    "PQ_chat_1",
    False,
    store,
    llm,
    "fast",
    "Disaster Recovery Plan.pdf",
    "PQ",
)
print(response.content["answer"])

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


[qa_chain] Found 4 conversation(s) for student 'student_1019'.
['COA_chat_1', 'COA_chat_2', 'COA_chat_3', 'SE_chat_1']
  [token_service] Tokens remaining for 'student_1019': 9,520,337
  [token_guard] Checking tokens — student: student_1019 | remaining: 9520337 | chain: run_qa_chain
[qa_chain] Starting QA — student: 'student_1019' | topic: 'Disaster Recovery Plan.pdf' | convo: 'PQ_chat_1' | pace: 'fast'
[qa_chain] New conversation 'PQ_chat_1' — STM initialised empty.
[qa_chain] New conversation 'PQ_chat_1' — STM initialised empty.
[qa_chain] No prior summary — first turn in this conversation.
[qa_chain] No LTM file found — starting fresh for student 'student_1019'.
[qa_chain] LTM: store is empty for student 'student_1019'.
[qa_chain] No STM context — reformulation skipped.
[Retriever] Phase 1 — Semantic search (top_k=5, threshold=0.4): 'I dislikes analogies and finds them condescending. I prefer '
Querying Chroma (top_k=5, threshold=0.4): 'Represent this sentence for searching relevant 

INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.


[qa_chain] QA LLM call succeeded. Grounded: None


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[qa_chain] STM summary updated (621 chars).
[qa_chain] STM saved for 'PQ_chat_1' (1 turns summarised).


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[qa_chain] LTM raw manager output: [ExtractedMemory(id='87721b9d-e094-4467-a4b0-cb84e8a6d39e', content=Memory(content='The student dislikes analogies and finds them condescending. They prefer straight to the point, technical explanations without illustrative comparisons. The student also prefers concise responses and gets disengaged when answers are padded with background context they did not ask for.')), ExtractedMemory(id='cf149e8c-338b-4efa-996d-6287d7bb18d6', content=Memory(content='The student consistently confuses RTO (Recovery Time Objective) and RPO (Recovery Point Objective), mixing them up across multiple past interactions.'))]
[qa_chain] LTM store saved for student 'student_1019' (2 memories).
[qa_chain] LTM: 2 new memory/memories extracted and saved.
[qa_chain] QA chain complete for student 'student_1019'.
  [token_service] Deducted 571 tokens — student: 'student_1019' | chain: run_qa_chain | in: 401 | out: 170 | remaining: 9,519,766
  [token_guard] ✓ Tokens deducted — stud

In [ ]:
# Obsereve the LLM's ltm about student
view_ltm("student_1019")

[qa_chain] LTM store loaded for student 'student_1019' (3 memories).


'Long-Term Memories for student_1019:\n────────────────────────────────────────\n1. The user dislikes the excessive use of analogies and prefers them to be avoided in future conversations.\n2. The user prefers explanations to be concise.\n3. The student prefers direct answers to their questions and dislikes unnecessary detail.'

### Quizz chain

In [4]:
from app.services.quiz_router import handle_quiz_request
import os 
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# Instantiate your llm (local or paid)

# llm = ChatOllama(model="mistral:latest")
load_dotenv()
api_key = os.environ.get('GOOGLE_API_KEY')

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", 
    project="denseless",       
    location="us-central1",              
    vertexai=True,
)

In [ ]:
# It supports time-travel testing (simulating a future date to test scheduled quizzes)
# Use the function below to interact with the quiz chain; The date must
# be in the format YYYY-MM-DD
result = handle_quiz_request(
    student_id      = "student_1019",
    course          = "CSC 403",
    topic           = "Software Engineering Intro",
    store           = store,
    llm             = llm,
    simulated_today = "2026-05-11"  # Optional: ISO date string
)
if result["status"] == "ok":
    quiz = result["quiz"]       # dict — 10 questions
else:
    print(result["message"])    # e.g. "Read your notes first."

NameError: name 'handle_quiz_request' is not defined

### Eval chain

In [ ]:
from app.agent.rag.chains.eval_chain import run_eval_chain
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

In [ ]:
# Instantiate your llm (local or paid)

# llm = ChatOllama(model="mistral:latest")
load_dotenv()
api_key = os.environ.get('GOOGLE_API_KEY')

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", 
    project="denseless",       
    location="us-central1",              
    vertexai=True,
)

In [ ]:
# Interact with the eval chain using the function below

response = run_eval_chain(
    student_id="student_1019",
    topic="Software Engineering Intro",
    quiz_phase="revision",
    quiz_path="../../data/quizzes/student_1019_software_engineering_intro_20260504_165955.json",
    llm=llm,
    simulated_date="2026-05-07"
)
print(response.content["questions"][0]["score"])
print(response.content["questions"][0]["explanation"])

  [token_service] Tokens remaining for 'student_1019': 9,724,160
  [token_guard] Checking tokens — student: student_1019 | remaining: 9724160 | chain: run_eval_chain
[eval_chain] Quiz loaded from ..\..\data\quizzes\student_1019_software_engineering_intro_20260504_165955.json
[eval_chain] Starting eval — student: 'student_1019' | topic: 'Software Engineering Intro' | phase: 'revision' | questions: 10
[eval_chain] Grading question 1/10...


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[eval_chain] Question graded. Score: 1.0
[eval_chain] Grading question 2/10...


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[eval_chain] Question graded. Score: 0.5
[eval_chain] Grading question 3/10...


...

[eval_chain] Question graded. Score: 1.0
[eval_chain] Grading question 10/10...


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[eval_chain] Question graded. Score: 0.5
[eval_chain] All 10 questions graded.
[eval_chain] Total score: 8.5 / 10
[eval_chain] Pre-resolution — strong: {'The Term Software Engineering', 'The Term Software', 'The Term Engineering', 'Computer Science VS Software Engineering'} | weak: {'The Term Software Engineering', 'The Term Engineering', 'Computer Science VS Software Engineering'}
[eval_chain] Post-resolution — strong: {'The Term Software'} | weak: {'The Term Software Engineering', 'The Term Engineering', 'Computer Science VS Software Engineering'}


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[eval_chain] Session feedback generated.
[eval_chain] Section classifications updated for topic 'Software Engineering Intro'.
[eval_chain] Revision score → retention (attempt 4).
[eval_chain] Revision date entry for 2026-05-07 marked completed.
[eval_chain] Graded quiz saved to ..\..\data\quizzes\student_1019_software_engineering_intro_20260504_165955.json
[eval_chain] Profile saved for student 'student_1019'.
[eval_chain] Eval chain complete for student 'student_1019'.
  [token_service] Deducted 6,551 tokens — student: 'student_1019' | chain: run_eval_chain | in: 6,011 | out: 540 | remaining: 9,717,609
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 6551 | remaining: 9717609
1.0
Correct — you identified both components accurately.
